# step back prompt

In [80]:
# %pip install mplfinance

In [81]:
import pandas as pd
import os

def read_taifex(path:str, product: str, product_m_or_w: str):
    '''
        讀取台灣期貨交易所的歷史資料，並選取特定商品代號及月份(週別)的資料

        年月日: str 
        商品代號: str
        商品月份: str

        回傳: DataFrame
    '''
    
    # 讀取 CSV 檔案，並將所有元素轉換成字串
    if os.path.exists(path):
        df = pd.read_csv(path, encoding="BIG5", dtype=str)
    else:
        return pd.DataFrame()
     
    # 移除所有元素的空白
    df = df.map(lambda x: x.strip())

    # 當 '到期月份(週別)' 包含 '/' 時，將 '成交價格' 的值改為 '近月價格' 的值
    df.loc[df['到期月份(週別)'].str.contains('/'), '成交價格'] = df['近月價格']
    
    # 到期月份(週別)中有些資料是202404/202405，代表轉倉，因此分割字串並選取第一個元素並存到新欄位 "商品月份"
    df['商品月份'] = df['到期月份(週別)'].str.split('/').str[0]

    df = df[df["商品代號"] == product]
    df = df[df["商品月份"] == product_m_or_w]

    # 將"交易日期" 和 "交易時間" 合併成一個新的欄位 "交易時間"
    df['Date'] = df['成交日期'] + df['成交時間']

    # 然後，將 "交易時間" 轉換成 datetime 對象
    # 假設 "交易日期" 是 "YYYYMMDD" 的格式，"交易時間" 是 "HHMMSS" 的格式
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d%H%M%S')

    df.set_index('Date', inplace=True)
    df = df.drop(columns=['成交日期'])
    df = df.drop(columns=['成交時間'])

    return df

In [82]:
import pandas as pd
import os
from datetime import datetime, time
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt
import mplfinance as mpf

# 假設你的 DataFrame 是 df
def to_kline(interval:str, df:pd.DataFrame):
    '''
    將原始交易資料轉換為 K 線資料

    參數:
        interval: K 線的時間間隔，例如 '5min' 代表 5 分鐘
        df: 原始交易資料，必須包含 '成交數量(B+S)' 和 '成交價格' 兩個欄位

    回傳值
        df_kline: K 線資料，包含 'Open', 'High', 'Low', 'Close', 'Volume' 五個欄位
    '''

    df.index = pd.to_datetime(df.index)  # 確保索引是 datetime 對象
    df['成交數量(B+S)'] = df['成交數量(B+S)'].astype(int)
    df['成交價格'] = df['成交價格'].astype(float)

    # 計算 OHLC
    df_ohlc = df['成交價格'].resample(interval).agg({'Open':'first', 'High':'max', 'Low':'min', 'Close':'last'})

    # 計算 Volume
    df_volume = df['成交數量(B+S)'].resample(interval).sum()

    # 將兩個 DataFrame 合併
    df_kline = pd.concat([df_ohlc, df_volume], axis=1)
    df_kline = df_kline.rename(columns={'成交數量(B+S)': 'Volume'})  # 將 '成交數量(B+S)' 重新命名為 'Volume'

    return df_kline

def fetch_time(datetime:datetime, time_start:time=time(8,45), time_end:time=time(13,45), interval:str="5min", symbol:str="MTX"):
    year = datetime.strftime('%Y')
    month = datetime.strftime('%m')
    day = datetime.strftime('%d')

    path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'history_data', 'tw', 'taifex', f'Daily_{year}_{month}_{day}.csv')
    
    # 讀取檔案，如果當天的資料不存在，則讀取下一個月的資料(轉倉後的資料)
    df = read_taifex(path, symbol, year + month)
    if df.empty:
        df = read_taifex(path, symbol, year + (datetime + relativedelta(months=1)).strftime('%m'))
    if df.empty:
        return None
    df_kline = to_kline(interval, df)

    # 轉成固定時間區間
    # df_kline.index = pd.to_datetime(df_kline.index)
    iday = df_kline[df_kline.index.date == datetime.date()]
    iday = iday[iday.index.time >= time_start]
    iday = iday[iday.index.time < time_end]

    return iday

def draw_kline(iday:pd.DataFrame):
    '''
        繪製 K 線圖
        參數:
            iday : pd.DataFrame : 股票資料
        回傳:
            kline_path : str : 圖片檔案路徑
    '''

    mc = mpf.make_marketcolors(up='r',down='g',inherit=True)
    s  = mpf.make_mpf_style(base_mpf_style='tradingview',marketcolors=mc)
    if int(iday.shape[0]) == 0:
        return None, None
    
    # 繪製 K 線圖
    fig, axes = mpf.plot(iday.dropna(), type='candle', style=s,
            ylabel='price',
            volume=True,
            returnfig=True)
    plt.close()

    kline_path = os.path.join("img", "fig.png")
    fig.savefig(kline_path)

    return kline_path

In [83]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import mplfinance as mpf
import google.generativeai as genai
from langchain.schema.messages import HumanMessage
from langchain.schema.messages import SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                convert_system_message_to_human=True,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

In [44]:
# k 線圖

from datetime import datetime, time
import matplotlib.pyplot as plt
import pandas as pd
import os
from datetime import timedelta
import json

# 創建日期範圍
start_date = "2024-01-01"
end_date = "2024-04-14"
date_range = pd.date_range(start=start_date, end=end_date)

# 創建時間範圍  9:00 ~ 13:00
time_range = [time(hour) for hour in range(9,14)]

# 對於每一天的每一小時調用 fetch_time
for date in date_range:
    for start_time in time_range:
        end_time = (datetime.combine(date, start_time) + timedelta(hours=1)).time()
        date_str = date.strftime("%Y%m%d")
        start_time_str = start_time.strftime("%H")
        end_time_str = end_time.strftime("%H")
        formatted_str = f"{date_str}_{start_time_str}_{end_time_str}"
        print(f"date: {date_str}, 從：{start_time_str}～{end_time_str}")
        df = fetch_time(date, start_time, end_time, "5min", "MTX")
        if df is None or len(df)<1:
            continue
        # df.to_csv("temp_data/" + formatted_str + ".csv")
        change = df['Close'].iloc[-1]/df['Open'].iloc[0]
        
        path = draw_kline(df)
        if path is None:
            continue


        # 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage
        messages = [
            SystemMessage(content="你是一個厲害的股市操盤手，專長是當日沖消"),
            
        ]

        # 生成第一次對話回應
        res1 = llm.invoke(messages)
        # print("res1", res1.content)

        # 將 AI 的回應添加到消息列表中
        messages.append(res1)
        messages.append(HumanMessage(content="k線對於市場情緒的概念是什麼?"))
        res2 = llm.invoke(messages)
        # print("res2", res2.content)

        # 將 AI 的回應添加到消息列表中
        messages.append(res2)
        messages.append(
            HumanMessage(
                content=[
                    {
                        "type": "text",
                        "text": "這是一張小台指的5分k線圖，請用上面的概念，並透過這張圖的具體形態，關鍵價格，成交量的變化趨勢，深入分析接下來一小時的市場情緒狀態，並猜測其隱含意義",
                    },
                    {"type": "image_url", "image_url": path}  # 替換為實際圖像路徑
                ]
            )
            HumanMessage(content="請用上面的概念，深入分析接下來一小時的市場情緒狀態"))
        res3 = llm.invoke(messages)
        print("res3", res3.content)
        
        res4 = llm.invoke(f"請依據這篇文章{res3.content}如果市場情緒好則只能回答 #long、如果不好則只能回答#short、如果不確定則只能回答 #unknown")
        sig_str = res4.content

        
        if sig_str.find("long") > 0:
            sig = 1
        elif sig_str.find("short")>0:
            sig = -1
        else:
            sig = 0
        print(sig_str, sig)

        output_data = {
            "date": date_str,
            "start_time": start_time_str,
            "end_time":end_time_str,
            "model": model,
            "temperature": temperature,
            "top_p": top_p,
            "seed": seed,
            "response_sig": sig,
            "change": change,
            "sig_str":sig_str,
        }
        json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
        out_path = os.path.join("out_kline_202401_05_2")
        os.makedirs(out_path, exist_ok=True)
        with open(out_path + "/" + formatted_str + ".json", 'w', encoding='utf-8') as f:
            f.write(json_content)
        


date: 20240415, 從：09～10
res3 根據提供的 5 分鐘 K 線圖，結合 K 線形態、成交量變化和技術指標，可以推測接下來一小時的市場情緒狀態，但需要強調的是，這僅僅是基於已有信息的推測，實際走勢會受到多種因素影響，不能完全依賴此分析。

**目前分析：**

* **市場情緒偏弱：** 價格在突破雙重頂頸線後，並未出現持續上漲，反而出現了一些回落。成交量也在突破後減少，說明市場交易熱度下降，買盤力量減弱。
* **潛在風險：** 價格如果跌破 20570 點，可能會進一步回落，測試下降趨勢線。如果跌破下降趨勢線，市場情緒可能更加悲觀，價格可能出現較大幅度的下跌。
* **潛在反彈機會：** 如果價格能夠站穩 20570 點，並突破下降趨勢線，市場情緒可能轉為樂觀，價格可能出現反彈。

**接下來一小時市場情緒推測：**

* **短線觀望：** 考慮到目前市場情緒偏弱，價格存在回落的風險，建議短線觀望，不要急於追漲。
* **關注關鍵價格：** 關注 20570 點和下降趨勢線的突破與否，以此判斷市場情緒變化。
* **觀察成交量：** 觀察成交量變化，如果成交量持續減少，說明市場交易熱度下降，可能不利於價格上漲。
* **參考技術指標：** 可以參考 MACD 和 RSI 等技術指標，進一步判斷市場情緒和力量。

**具體操作：**

* **如果價格跌破 20570 點，** 可以考慮做空，但要設定止損點。
* **如果價格站穩 20570 點並突破下降趨勢線，** 可以考慮做多，但要謹慎觀察成交量和技術指標。
* **如果價格在 20570 點和下降趨勢線之間震盪，** 可以選擇觀望，等待明確的突破或跌破信號。

**再次強調，以上僅為個人觀點，不构成任何投资建议。投資者應自行判断风险，并参考其他信息进行决策。**



#short 
 -1
date: 20240415, 從：10～11


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.


res3 根據你提供的圖表，我可以推測接下來一小時的市場情緒狀態，但請注意，這只是我的分析，並非投資建議。

**觀察到的市場情緒跡象：**

* **價格下跌：** 價格在過去的5分鐘K線圖中一直呈現下跌趨勢，這表明賣方力量較強，市場情緒偏向悲觀。
* **成交量增加：** 10:40附近的成交量明顯增加，這表明市場參與者對市場走向有較大的分歧，交易活躍度高，但由於價格仍在下降，這可能表明賣方力量更強。
* **K線形態：** 10:50的長紅K線，顯示買方力量出現短暫反彈，但價格很快再次下跌，表明這種反彈力量不足，市場情緒仍然偏向悲觀。

**接下來一小時的市場情緒推測：**

* **短期內，市場情緒可能仍然偏向悲觀：** 因為價格下跌趨勢尚未扭轉，成交量也表明賣方力量強勢。
* **可能出現反彈：** 20520附近的支撐位是買方力量的最後防線，如果價格跌破該支撐位，市場情緒可能會變得更加悲觀，但如果價格在該支撐位附近獲得支撐，則可能會出現反彈。
* **觀望為主：** 在這種情況下，建議採取觀望策略，不要盲目追漲或殺跌。

**需要關注的因素：**

* **消息面：** 一小時內可能出現的任何重大消息，例如經濟數據公布或政策調整，都會對市場情緒產生重大影響。
* **其他技術指標：** 除了K線之外，還可以參考其他技術指標，例如MACD、RSI等，以更全面地了解市場情緒。
* **市場心理：** 市場情緒也會受到投資者心理的影響，例如恐慌情緒、貪婪情緒等。

**結論：**

總體而言，根據目前的情況，接下來一小時的市場情緒可能仍然偏向悲觀，但有反彈的可能性。建議投資者密切關注市場動態，採取謹慎的投資策略。

**再次強調，以上分析僅供參考，不構成任何投資建議。** 投資者應該根據自己的風險承受能力和投資目標，做出自己的投資決策。

#short 
 -1
date: 20240415, 從：11～12
res3 基於提供的 5 分鐘 K 線圖，以下分析接下來一小時的市場情緒狀態：

**利多因素：**

* **上升趨勢:** 價格持續上漲，顯示多頭力量強勁，市場情緒傾向於樂觀。
* **突破阻力:** 價格已突破 20510 點的阻力位，顯示多頭力量強勁，未來有望繼續上漲。
* **成交量放大:** 11:40 到 11:50 的成交量放大，顯示市場

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.


res3 根據你提供的 5 分鐘 K 線圖，以及我們對 K 線圖對於市場情緒的理解，接下來一小時的市場情緒狀態，可以從以下幾個方面分析：

**1. 價格走勢:**

* 圖表呈現明顯的下跌趨勢，賣方力量強勁，價格多次嘗試反彈，但都未能突破阻力位，顯示賣方力量佔優勢。
* 19920 點是重要的阻力位，如果價格無法突破此阻力位，預期接下來一小時的市場情緒會繼續偏向悲觀。

**2. 成交量:**

* 成交量持續保持在高位，顯示市場參與者對價格走勢感到興趣，但成交量並沒有明顯放大，表示市場情緒仍然偏向觀望，並沒有出現明顯的恐慌性拋售。

**3. 蠟燭圖的整體形態:**

* 價格走勢呈現連續下跌，顯示市場情緒悲觀，但蠟燭圖的實體並不算特別長，表示市場情緒仍然相對穩定，並沒有出現明顯的恐慌性拋售。

**4. 市場情緒的預測:**

* 基於以上分析，接下來一小時的市場情緒預期會繼續偏向悲觀，但由於成交量沒有明顯放大，市場情緒仍然相對穩定，預期價格會繼續在一定區間內波動。

**5.  其他需要觀察的因素:**

* 需要觀察是否有新的消息面或事件出現，可能會影響市場情緒。
* 需要觀察其他技術指標，例如移動平均線、MACD 等，以確認市場情緒的變化。

**需要注意的是:**

* 以上分析僅供參考，不構成任何投資建議。
* 市場情緒瞬息萬變，可能受到各種因素影響，實際走勢可能會與預期有所偏差。
* 投資者應根據自己的風險承受能力和交易策略，做出合理的交易決策。

**總而言之:** 雖然市場情緒目前偏向悲觀，但由於成交量沒有明顯放大，市場情緒仍然相對穩定，預期價格會繼續在一定區間內波動。接下來一小時的市場走勢，需要觀察是否有新的消息面或事件出現，以及其他技術指標的變化。

希望以上分析可以幫助你更好地了解市場情緒，並做出更合理的交易決策。

#short 
 -1
date: 20240416, 從：12～13
res3 根據提供的圖表，接下來一小時的市場情緒狀態，可以從以下幾個方面深入分析：

**1.  整體趨勢:**

* 價格整體呈現下跌趨勢，吞噬形態和整理形態都暗示空頭力量較強。
* 成交量逐步減少，說明市場觀望情緒濃厚，價格短期內可能維持震盪走勢。

**2.  市場情緒的關鍵轉折點:**

* 19870 點是吞噬形態的頸線位置，如果價格跌破此點，

...

#unknown 
 0
date: 20240422, 從：13～14
res3 根据提供的5分钟K线图，可以分析出接下来一小时的市场情绪状态可能存在以下几种情况：

**1. 持续看跌：**

* 13:40的K线跳空下跌，显示空头力量强劲，市场情绪恐慌，可能进一步加剧看跌情绪。
* 13:30-13:40的红K线实体较大，成交量放大，代表卖盘积极，空头力量占优。
* 若接下来一小时内出现持续下跌的K线，并且成交量维持高位，则可以判断市场情绪持续看跌，价格可能进一步下跌。

**2. 震荡调整：**

* 13:30-13:40的红K线实体较大，但成交量略有下降，代表卖盘力量虽然强劲，但市场参与度有所下降。
* 13:40的K线跳空下跌，但幅度不大，代表市场情绪虽然恐慌，但没有完全崩溃。
* 若接下来一小时内出现小幅波动，并且成交量保持低位，则可以判断市场情绪处于震荡调整阶段，价格可能在一定范围内波动。

**3. 短期反弹：**

* 13:40的K线跳空下跌，但价格距离重要支撑位19500点并不远，存在反弹的可能性。
* 13:30-13:40的红K线实体较大，成交量放大，代表卖盘积极，但同时也可能存在短线获利回吐的因素。
* 若接下来一小时内出现小幅反弹，并且成交量有所下降，则可以判断市场情绪可能出现短期反弹，价格可能短线反弹至19500点附近。

**需要注意的是：**

* 以上分析仅是基于现有信息进行的推测，实际市场走势可能会受到其他因素影响，比如突发事件、政策变化等。
* 市场情绪变化很快，可能会在短时间内发生逆转。
* 投资有风险，入市需谨慎。

建议您结合其他指标和信息，并根据自己的风险承受能力，做出合理的投资决策。

#unknown 
 0
date: 20240423, 從：09～10
res3 好的，讓我們根據之前的K線圖，以及市場情緒的相關概念，深入分析接下來一小時的市場情緒狀態。

**回顾：**

* 9點到9點10分：價格連續下跌，成交量逐漸減少，顯示市場參與者對於價格的信心不足，情緒偏向悲觀。
* 9點10分到9點20分：價格繼續下跌，但成交量大幅減少，顯示市場參與者觀望態度，情緒趨向平淡。
* 9點20分到9點30分：價格出現反彈的紅色K棒，成交量開始增加，顯示市場參與者對於價格反彈的興趣，情緒開始轉向樂觀。
* 9點30分

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.


#unknown 

根據文章的分析，市場呈現震盪整理態勢，成交量逐漸減少，收盤價低於開盤價，預計接下來一小時可能出現小幅反彈。 

雖然預測市場情緒偏向悲觀，但跌幅較小，有小幅反彈的可能性，但實際走勢可能與預期有所偏差。

因此，無法明確判斷市場情緒是好還是不好，只能回答 #unknown。 
 0
date: 20240430, 從：13～14
res3 根据你提供的图表，结合K线和成交量的变化，可以推断出接下来一小时的市场情绪状态，但需要谨慎，因为市场瞬息万变，无法完全预测：

**1.  总体情绪：**

* 当前市场情绪偏向悲观，价格呈现下跌趋势，并且突破了头肩顶形态，这是一个明显的看跌信号。
* 成交量放大，表明市场参与者对下跌趋势的认可度增加，也反映出市场情绪的波动性。

**2.  短期预期：**

* 短期内，市场情绪可能继续维持悲观，价格有可能继续下跌。
* 跌破20420点支撑位，下跌趋势可能会进一步加剧，市场情绪将更加悲观。
* 若价格反弹至20480点颈线附近，可能会出现短期的反弹，但反弹的持续性可能有限，市场情绪可能会出现短暂的乐观，但很快就会再次转为悲观。

**3.  影响因素：**

*  市场消息面：接下来的一个小时，如果出现利好消息，可能会短暂提振市场情绪，导致价格反弹；反之，如果出现利空消息，则会加剧市场悲观情绪，导致价格继续下跌。
*  技术指标：其他技术指标，如MACD、RSI等，也能提供市场情绪的参考信息，需要综合分析。
*  资金流向：资金流向可以反映市场情绪的变化，例如，如果大量资金流出，则市场情绪可能会更加悲观。

**4.  交易策略：**

*  基于当前的市场情绪，考虑逢高做空，但需要设置好止损位，控制风险。
*  关注20420点支撑位和20480点颈线，根据价格突破或反弹情况调整交易策略。

**总结：**

接下来一小时的市场情绪可能继续维持悲观，价格有可能继续下跌。但需要密切关注市场消息面、技术指标和资金流向等因素，及时调整交易策略，控制风险。

**再次强调，以上分析仅供参考，不构成投资建议。市场瞬息万变，请自行判断风险。**


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.


#short 
 -1


In [79]:
import pandas as pd
import json

# 計算準確率和平均報酬率
gain = []
loss = []

tp = 0
fp = 0
tn = 0
fn = 0

files = os.listdir("out_kline_202401_05_2")
files = sorted(files)

for idx, file in enumerate(files):
    if file.split("_")[1] != "09" and file.split("_")[1] != "10" and file.split("_")[1] != "11" and file.split("_")[1] != "12":
        continue
    # print(file)
    with open("out_kline_202401_05_2/" + file, "r", encoding='utf-8') as f:
        data = json.load(f)
        sig = data["response_sig"]
        # print(data["response_sig"], data["sig_str"])

    if idx+1 == len(files):
        break
    # print(files[idx+1])
    with open("out_kline_202401_05_2/" + files[idx+1], "r", encoding='utf-8') as f:
        data = json.load(f)
        rtn = data["change"]
        rtn = float(rtn) - 1
        # print(data["change"])
    # print()
    
    if pd.isna(rtn):
        rtn = 0

    if sig > 0:
        if rtn > 0:
            tp += 1
            gain.append(rtn)
        else:
            fp += 1
            loss.append(rtn)
    elif sig == -1: 
        if rtn < 0:
            tn += 1
            gain.append(-rtn)
        else:
            fn += 1
            loss.append(-rtn)



accuracy = (tp + tn) / (tp + fp + tn + fn)
print(tn+tp+fn+fp)
print(f'Accuracy: {accuracy}')
print(f'Average gain: {sum(gain) / len(gain)}')
print(f'Average loss: {sum(loss) / len(loss)}')
print(f'expected value: {(sum(gain) / len(gain))*accuracy + (sum(loss) / len(loss))*(1-accuracy)}')

15
Accuracy: 0.6
Average gain: 0.0012927254325676059
Average loss: -0.0014412879321668386
expected value: 0.00019912008667382806
